# HW 6 Dedupe

Questo notebook implementa il punto **4.E**, **4.F** e **4.H** dell'Homework 6 utilizzando la libreria **Dedupe**.

### Caratteristiche e Scelte Metodologiche Principali:
1. **Riuso degli Artifact Comuni (Zero Data Leakage)**: Lettura dei dataset parquet e delle coppie etichettate (rapporto 1:5) generati in `HW_6_GT_R_2.ipynb`, con vincolo formale di assenza del VIN.
2. **Feature di Confronto Dedupe**: Selezione accurata dei campi descrittivi (`make`, `model`, `year`, `price`) gestendo valori nulli (`has_missing=True`).
3. **Addestramento One-Off del Modello**: Addestramento del classificatore probabilistico di Dedupe su `train_pairs_1_5.csv` una sola volta sul set di training; B1 e B2 rappresentano due strategie di candidate generation applicate al medesimo modello addestrato.
4. **Candidate Generation B1 e B2 Allineata**: Blocco esatto su `make` (B1, ~42.494 coppie) e *Sorted Neighbourhood* su `year` con `window=1` (B2, ~46.870 coppie).
5. **Selezione della Soglia Esclusivamente su Validation ($th^*$)**: Ricerca a griglia (coarse + fine) per massimizzare l'$F_1$-score sul validation set, con congelamento della soglia prima della fase di test.
6. **Risoluzione Conflitti 1:1 e Misurazione Omogenea dell'Inferenza**: Risoluzione globale dei duplicati tramite `linker.one_to_one()` e scomposizione cronometrica rigorosa in $T_{\text{blocking}}$, $T_{\text{scoring}}$ e $T_{\text{selection}}$ ($T_{\text{matching}}$), del tutto speculare alla pipeline Ditto.

##Configurazione Ambiente e Dipendenze


In [1]:
# BTrees<6 evita problemi di compatibilità con Dedupe 3.0.2.
%pip install -q "dedupe==3.0.2" "BTrees<6.0" "recordlinkage==0.16" loguru

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.9/106.9 kB 5.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.8/201.8 kB 15.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.1/134.1 kB 15.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 926.9/926.9 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os
import sys
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import recordlinkage
import dedupe
from dedupe import variables
from loguru import logger

os.environ['TZ'] = 'Europe/Rome'
time.tzset()

logger.remove()
logger.add(
    sys.stdout,
    colorize=True,
    format="<green>{time:YYYY-MM-DD HH:mm:ss}</green> | <level>{level: <8}</level> | <cyan>{message}</cyan>"
)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

ARTIFACT_DIR = Path("/content/drive/MyDrive/data")
if not ARTIFACT_DIR.exists() and Path("/content/drive/MyDrive/ing_dati/data").exists():
    ARTIFACT_DIR = Path("/content/drive/MyDrive/ing_dati/data")

#parametri Dedupe per l'inizializzazione del learner attivo
DEDUPE_SAMPLE_SIZE = 5000
INTERNAL_BLOCKING_RECALL = 0.90

logger.info(f"Directory artifact di riferimento: {ARTIFACT_DIR}")

2026-09-07 00:58:13 | INFO     | Directory artifact di riferimento: /content/drive/MyDrive/data


##Caricamento degli Artifact da Google Drive (Privi di VIN)


In [4]:
required_files = [
    "ground_truth.csv",
    "train_gt.csv",
    "validation_gt.csv",
    "test_gt.csv",
    "train_pairs_1_5.csv",
    "val_pairs_1_5.csv",
    "test_pairs_1_5.csv",
    "vehicles_train.parquet",
    "used_cars_train.parquet",
    "vehicles_validation.parquet",
    "used_cars_validation.parquet",
    "vehicles_test.parquet",
    "used_cars_test.parquet",
]

missing_files = [
    name for name in required_files
    if not (ARTIFACT_DIR / name).exists()
]

if missing_files:
    raise FileNotFoundError(
        "Mancano questi artifact. Esegui prima HW_6_GT_R_2:\n- "
        + "\n- ".join(missing_files)
    )

logger.success("Tutti gli artifact necessari sono presenti su Google Drive.")

2026-09-07 00:58:14 | SUCCESS  | Tutti gli artifact necessari sono presenti su Google Drive.


In [5]:
ground_truth = pd.read_csv(ARTIFACT_DIR / "ground_truth.csv")

train_gt = pd.read_csv(ARTIFACT_DIR / "train_gt.csv")
validation_gt = pd.read_csv(ARTIFACT_DIR / "validation_gt.csv")
test_gt = pd.read_csv(ARTIFACT_DIR / "test_gt.csv")

#coppie etichettate generate nel notebook GT
train_pairs_df = pd.read_csv(ARTIFACT_DIR / "train_pairs_1_5.csv")
val_pairs_df = pd.read_csv(ARTIFACT_DIR / "val_pairs_1_5.csv")
test_pairs_df = pd.read_csv(ARTIFACT_DIR / "test_pairs_1_5.csv")

vehicles_train = pd.read_parquet(
    ARTIFACT_DIR / "vehicles_train.parquet"
).set_index("orig_cl_id", drop=False)

used_cars_train = pd.read_parquet(
    ARTIFACT_DIR / "used_cars_train.parquet"
).set_index("orig_uc_id", drop=False)

vehicles_val = pd.read_parquet(
    ARTIFACT_DIR / "vehicles_validation.parquet"
).set_index("orig_cl_id", drop=False)

used_cars_val = pd.read_parquet(
    ARTIFACT_DIR / "used_cars_validation.parquet"
).set_index("orig_uc_id", drop=False)

vehicles_test = pd.read_parquet(
    ARTIFACT_DIR / "vehicles_test.parquet"
).set_index("orig_cl_id", drop=False)

used_cars_test = pd.read_parquet(
    ARTIFACT_DIR / "used_cars_test.parquet"
).set_index("orig_uc_id", drop=False)

#cerenza con il notebook Record Linkage, year trattato come stringa
for df in [
    vehicles_train, used_cars_train,
    vehicles_val, used_cars_val,
    vehicles_test, used_cars_test
]:
    df["year"] = df["year"].astype("string")

logger.info(f"TRAIN      : {len(vehicles_train):,d} (cl) / {len(used_cars_train):,d} (uc) | GT: {len(train_gt):,d}")
logger.info(f"VALIDATION : {len(vehicles_val):,d} (cl) / {len(used_cars_val):,d} (uc) | GT: {len(validation_gt):,d}")
logger.info(f"TEST       : {len(vehicles_test):,d} (cl) / {len(used_cars_test):,d} (uc) | GT: {len(test_gt):,d}")

2026-09-07 00:58:40 | INFO     | TRAIN      : 2,368 (cl) / 2,368 (uc) | GT: 2,368
2026-09-07 00:58:40 | INFO     | VALIDATION : 789 (cl) / 789 (uc) | GT: 789
2026-09-07 00:58:40 | INFO     | TEST       : 790 (cl) / 790 (uc) | GT: 790


##Controlli di Coerenza e Prevenzione Data Leakage


In [6]:
def pair_set(df):
    return set(
        zip(
            df["orig_cl_id"].astype(int),
            df["orig_uc_id"].astype(int)
        )
    )

def check_pair_file(name, pairs_df, gt_df):
    required = {"orig_cl_id", "orig_uc_id", "label"}
    missing = required - set(pairs_df.columns)

    if missing:
        raise ValueError(
            f"{name}: colonne mancanti {sorted(missing)}"
        )

    n_pos = int((pairs_df["label"] == 1).sum())
    n_neg = int((pairs_df["label"] == 0).sum())

    positives = set(
        zip(
            pairs_df.loc[pairs_df["label"] == 1, "orig_cl_id"].astype(int),
            pairs_df.loc[pairs_df["label"] == 1, "orig_uc_id"].astype(int)
        )
    )

    expected_positives = pair_set(gt_df)

    if positives != expected_positives:
        raise ValueError(
            f"{name}: i positivi non coincidono con la GT dello split."
        )

    if n_pos == 0 or n_neg == 0:
        raise ValueError(f"{name}: servono sia positivi sia negativi.")

    logger.info(
        f"{name:18s} | {n_pos:,d} positivi | {n_neg:,d} negativi | "
        f"rapporto neg/pos = {n_neg / n_pos:.2f}"
    )

check_pair_file("TRAIN PAIRS", train_pairs_df, train_gt)
check_pair_file("VALIDATION PAIRS", val_pairs_df, validation_gt)
check_pair_file("TEST PAIRS", test_pairs_df, test_gt)

2026-09-07 00:58:40 | INFO     | TRAIN PAIRS        | 2,368 positivi | 11,840 negativi | rapporto neg/pos = 5.00
2026-09-07 00:58:40 | INFO     | VALIDATION PAIRS   | 789 positivi | 3,945 negativi | rapporto neg/pos = 5.00
2026-09-07 00:58:40 | INFO     | TEST PAIRS         | 790 positivi | 3,950 negativi | rapporto neg/pos = 5.00


In [7]:
#nessun VIN deve arrivare a Dedupe (consegna 4.B)
for name, df in [
    ("vehicles_train", vehicles_train),
    ("used_cars_train", used_cars_train),
    ("vehicles_val", vehicles_val),
    ("used_cars_val", used_cars_val),
    ("vehicles_test", vehicles_test),
    ("used_cars_test", used_cars_test),
]:
    if "vin" in [str(c).lower() for c in df.columns]:
        raise ValueError(f"VIN ancora presente in {name}")

logger.success("Verifica superata: attributo VIN rigorosamente assente da tutti gli split.")

2026-09-07 00:58:40 | SUCCESS  | Verifica superata: attributo VIN rigorosamente assente da tutti gli split.


In [8]:
#controllo formale assenza leakage tra split
cl_train_ids = set(train_gt["orig_cl_id"].astype(int))
cl_val_ids = set(validation_gt["orig_cl_id"].astype(int))
cl_test_ids = set(test_gt["orig_cl_id"].astype(int))

uc_train_ids = set(train_gt["orig_uc_id"].astype(int))
uc_val_ids = set(validation_gt["orig_uc_id"].astype(int))
uc_test_ids = set(test_gt["orig_uc_id"].astype(int))

overlaps = {
    "CL train-val": len(cl_train_ids & cl_val_ids),
    "CL train-test": len(cl_train_ids & cl_test_ids),
    "CL val-test": len(cl_val_ids & cl_test_ids),
    "UC train-val": len(uc_train_ids & uc_val_ids),
    "UC train-test": len(uc_train_ids & uc_test_ids),
    "UC val-test": len(uc_val_ids & uc_test_ids),
}

if any(v != 0 for v in overlaps.values()):
    raise ValueError(
        f"Leakage tra split rilevato: {overlaps}"
    )

logger.success(f"Split rigorosamente disgiunti (Zero Leakage): {overlaps}")

2026-09-07 00:58:40 | SUCCESS  | Split rigorosamente disgiunti (Zero Leakage): {'CL train-val': 0, 'CL train-test': 0, 'CL val-test': 0, 'UC train-val': 0, 'UC train-test': 0, 'UC val-test': 0}


In [9]:
#verifica se la Ground Truth è realmente 1-to-1
all_gt = pd.concat(
    [train_gt, validation_gt, test_gt],
    ignore_index=True
).drop_duplicates(["orig_cl_id", "orig_uc_id"])

GT_IS_ONE_TO_ONE = (
    not all_gt["orig_cl_id"].duplicated().any()
    and not all_gt["orig_uc_id"].duplicated().any()
)

logger.info(f"Ground Truth formalmente 1-to-1: {GT_IS_ONE_TO_ONE}")
if not GT_IS_ONE_TO_ONE:
    logger.warning(
        "La GT non è 1-to-1: in valutazione verrà usata una semplice soglia many-to-many."
    )

2026-09-07 00:58:40 | INFO     | Ground Truth formalmente 1-to-1: True


### Nota Metodologica sui file `val_pairs_1_5.csv` e `test_pairs_1_5.csv`

Vengono caricati e verificati per completezza, ma **non vengono usati per calcolare le metriche finali di Dedupe**.

Il motivo è rigoroso: quei file contengono un campione artificiale bilanciato di negativi 1:5. Per misurare correttamente Precision, Recall e F1 della pipeline reale **B1-Dedupe / B2-Dedupe**, è indispensabile:
1. Generare tutte le candidate pairs ammesse dalle regole di blocking B1 o B2 sullo split;
2. Effettuare lo scoring probabilistico con Dedupe;
3. Risolvere i conflitti 1:1 e confrontare le predizioni con la totalità della Ground Truth dello split.


##Conversione dei Record nel Formato Dizionario di Dedupe


In [10]:
def safe_text(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    if value == "":
        return None

    return value


def safe_price(value):
    if pd.isna(value):
        return None

    try:
        value = float(value)
    except (TypeError, ValueError):
        return None

    # dedupe.variables.Price richiede numeri strettamente positivi.
    if not np.isfinite(value) or value <= 0:
        return None

    return value


def create_dedupe_dict(df, id_col):
    price_col = "price_num" if "price_num" in df.columns else "price"

    required = {
        id_col,
        "make",
        "model",
        "year",
        price_col
    }

    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"Colonne mancanti in dataset: {sorted(missing)}"
        )

    data = {}

    for _, row in df.iterrows():
        record_id = int(row[id_col])

        data[record_id] = {
            "make": safe_text(row["make"]),
            "model": safe_text(row["model"]),
            "year": safe_text(row["year"]),
            "price": safe_price(row[price_col]),
        }

    return data

In [11]:
cl_train_dict = create_dedupe_dict(
    vehicles_train, "orig_cl_id"
)
uc_train_dict = create_dedupe_dict(
    used_cars_train, "orig_uc_id"
)

cl_val_dict = create_dedupe_dict(
    vehicles_val, "orig_cl_id"
)
uc_val_dict = create_dedupe_dict(
    used_cars_val, "orig_uc_id"
)

cl_test_dict = create_dedupe_dict(
    vehicles_test, "orig_cl_id"
)
uc_test_dict = create_dedupe_dict(
    used_cars_test, "orig_uc_id"
)

logger.info(f"Dizionari Dedupe TRAIN      : {len(cl_train_dict):,d} (cl) | {len(uc_train_dict):,d} (uc)")
logger.info(f"Dizionari Dedupe VALIDATION : {len(cl_val_dict):,d} (cl) | {len(uc_val_dict):,d} (uc)")
logger.info(f"Dizionari Dedupe TEST       : {len(cl_test_dict):,d} (cl) | {len(uc_test_dict):,d} (uc)")

2026-09-07 00:58:41 | INFO     | Dizionari Dedupe TRAIN      : 2,368 (cl) | 2,368 (uc)
2026-09-07 00:58:41 | INFO     | Dizionari Dedupe VALIDATION : 789 (cl) | 789 (uc)
2026-09-07 00:58:41 | INFO     | Dizionari Dedupe TEST       : 790 (cl) | 790 (uc)


##Costruzione delle Coppie Etichettate per il Training (Rapporto 1:5)


In [12]:
def labeled_pairs_from_dataframe(
    pairs_df,
    cl_data,
    uc_data
):
    labeled = {
        "match": [],
        "distinct": []
    }

    missing_ids = []

    for row in pairs_df.itertuples(index=False):
        cl_id = int(row.orig_cl_id)
        uc_id = int(row.orig_uc_id)
        label = int(row.label)

        if cl_id not in cl_data or uc_id not in uc_data:
            missing_ids.append((cl_id, uc_id))
            continue

        record_pair = (
            cl_data[cl_id],
            uc_data[uc_id]
        )

        if label == 1:
            labeled["match"].append(record_pair)
        elif label == 0:
            labeled["distinct"].append(record_pair)
        else:
            raise ValueError(f"Label non valida: {label}")

    if missing_ids:
        raise ValueError(
            f"{len(missing_ids)} coppie di training fanno riferimento "
            "a record non presenti negli split."
        )

    return labeled

labeled_pairs = labeled_pairs_from_dataframe(
    train_pairs_df,
    cl_train_dict,
    uc_train_dict
)

logger.info(f"Coppie etichettate per training | Positivi (match): {len(labeled_pairs['match']):,d} | Negativi (distinct): {len(labeled_pairs['distinct']):,d}")

2026-09-07 00:58:41 | INFO     | Coppie etichettate per training | Positivi (match): 2,368 | Negativi (distinct): 11,840


Per le etichette già esistenti utilizziamo un **training file Dedupe** (`dedupe_training_1_5.json`).
È la modalità ufficiale raccomandata da Dedupe quando si possiede già un dataset supervisionato: le coppie 1:5 vengono caricate durante `prepare_training`, garantendo riproducibilità e rapidità.


In [13]:
DEDUPE_TRAINING_FILE = ARTIFACT_DIR / "dedupe_training_1_5.json"

with open(DEDUPE_TRAINING_FILE, "w", encoding="utf-8") as f:
    dedupe.write_training(labeled_pairs, f)

logger.success(f"Training file Dedupe esportato con successo in: {DEDUPE_TRAINING_FILE}")

2026-09-07 00:58:42 | SUCCESS  | Training file Dedupe esportato con successo in: /content/drive/MyDrive/data/dedupe_training_1_5.json


##Definizione delle Feature e Addestramento del Modello Dedupe


In [14]:
fields = [
    variables.String(
        "make",
        has_missing=True
    ),
    variables.String(
        "model",
        has_missing=True
    ),
    variables.ShortString(
        "year",
        has_missing=True
    ),
    variables.Price(
        "price",
        has_missing=True
    ),
]

# num_cores=0 rende lo scoring più robusto in notebook
# evitando problemi legati al multiprocessing.
linker = dedupe.RecordLink(
    fields,
    num_cores=0
)

In [15]:
start_train = time.perf_counter()

with open(
    DEDUPE_TRAINING_FILE,
    "r",
    encoding="utf-8"
) as training_file:
    linker.prepare_training(
        cl_train_dict,
        uc_train_dict,
        training_file=training_file,
        sample_size=DEDUPE_SAMPLE_SIZE,
        blocked_proportion=0.9,
    )

#le regole B1/B2 sono blocking ESTERNI allineati alle altre pipeline.
#disattiviamo gli index predicates interni per evitare NoIndexError e garantire indipendenza.
linker.train(
    recall=INTERNAL_BLOCKING_RECALL,
    index_predicates=False
)

dedupe_training_time = time.perf_counter() - start_train

logger.success(f"Addestramento classificatore Dedupe completato in: {dedupe_training_time:.2f} s")

2026-09-07 01:05:44 | SUCCESS  | Addestramento classificatore Dedupe completato in: 422.63 s


> [!NOTE]
> **Importante**: Dedupe apprende il classificatore probabilistico (regressione logistica su distanze stringa/prezzo) dalle coppie etichettate. Le regole di blocking interne generate da `train()` non vengono usate per B1/B2: in inferenza passiamo esplicitamente a `score()` le candidate pairs prodotte dalle due strategie definite nell'Homework.


##Candidate Generation per Blocking B1 (Make) e B2 (Year)


In [16]:
def create_candidates(df_cl, df_uc, strategy):
    start = time.perf_counter()

    indexer = recordlinkage.Index()

    if strategy == "B1":
        # B1: blocking esatto sulla marca.
        indexer.block(
            left_on="make",
            right_on="make"
        )

    elif strategy == "B2":
        # B2: stessa scelta del notebook Record Linkage.
        indexer.sortedneighbourhood(
            left_on="year",
            right_on="year",
            window=1
        )

    else:
        raise ValueError(
            "strategy deve essere 'B1' oppure 'B2'"
        )

    candidates = indexer.index(df_cl, df_uc)

    blocking_time = time.perf_counter() - start

    return candidates, blocking_time


def blocking_metrics(candidates, gt, n_left, n_right):
    true_pairs = pair_set(gt)
    candidate_set = set(
        (int(a), int(b))
        for a, b in candidates
    )

    captured = len(
        true_pairs & candidate_set
    )

    recall = (
        captured / len(true_pairs)
        if true_pairs else 0.0
    )

    cartesian = n_left * n_right

    reduction_ratio = (
        1 - len(candidates) / cartesian
        if cartesian else 0.0
    )

    return {
        "Candidate Pairs": len(candidates),
        "True matches captured": captured,
        "Blocking Recall": recall,
        "Reduction Ratio": reduction_ratio,
    }

##Scoring delle Candidate Pairs con Dedupe


In [17]:
def iter_candidate_records(
    candidates,
    cl_data,
    uc_data
):
    for cl_id, uc_id in candidates:
        cl_id = int(cl_id)
        uc_id = int(uc_id)

        yield (
            (cl_id, cl_data[cl_id]),
            (uc_id, uc_data[uc_id])
        )


def score_candidates(
    linker,
    candidates,
    cl_data,
    uc_data
):
    start = time.perf_counter()

    scores = linker.score(
        iter_candidate_records(
            candidates,
            cl_data,
            uc_data
        )
    )

    scoring_time = time.perf_counter() - start

    return scores, scoring_time

##Risoluzione Conflitti 1:1 e Calcolo Metriche di Entity Resolution


In [18]:
def prediction_set_from_scores(
    linker,
    scores,
    threshold
):
    threshold = float(threshold)

    if GT_IS_ONE_TO_ONE:
        links = linker.one_to_one(
            scores,
            threshold=threshold
        )

        return {
            (int(pair[0]), int(pair[1]))
            for pair, score in links
        }

    #fallback many-to-many se la GT non fosse one-to-one.
    return {
        (int(row["pairs"][0]), int(row["pairs"][1]))
        for row in scores
        if float(row["score"]) > threshold
    }


def evaluate_predictions(
    predicted_pairs,
    gt
):
    predicted = set(predicted_pairs)
    truth = pair_set(gt)

    tp = len(predicted & truth)
    fp = len(predicted - truth)
    fn = len(truth - predicted)

    precision = (
        tp / (tp + fp)
        if tp + fp else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
    }


def metrics_at_threshold(
    linker,
    scores,
    gt,
    threshold
):
    predicted = prediction_set_from_scores(
        linker,
        scores,
        threshold
    )

    return evaluate_predictions(
        predicted,
        gt
    )

##Validation Set: Costruzione Candidati e Scoring B1/B2


In [19]:
#B1 validation (Make)
val_candidates_B1, val_block_time_B1 = create_candidates(
    vehicles_val,
    used_cars_val,
    "B1"
)
val_block_stats_B1 = blocking_metrics(
    val_candidates_B1,
    validation_gt,
    len(vehicles_val),
    len(used_cars_val)
)
val_scores_B1, val_score_time_B1 = score_candidates(
    linker,
    val_candidates_B1,
    cl_val_dict,
    uc_val_dict
)

#B2 validation (Year)
val_candidates_B2, val_block_time_B2 = create_candidates(
    vehicles_val,
    used_cars_val,
    "B2"
)
val_block_stats_B2 = blocking_metrics(
    val_candidates_B2,
    validation_gt,
    len(vehicles_val),
    len(used_cars_val)
)
val_scores_B2, val_score_time_B2 = score_candidates(
    linker,
    val_candidates_B2,
    cl_val_dict,
    uc_val_dict
)

logger.info(f"[Validation B1 - Make] Candidati: {val_block_stats_B1['Candidate Pairs']:,d} | Catturati: {val_block_stats_B1['True matches captured']:,d}/{len(validation_gt):,d} | Recall: {val_block_stats_B1['Blocking Recall']:.4f} | Tempo: {val_block_time_B1:.3f}s")
logger.info(f"[Validation B2 - Year] Candidati: {val_block_stats_B2['Candidate Pairs']:,d} | Catturati: {val_block_stats_B2['True matches captured']:,d}/{len(validation_gt):,d} | Recall: {val_block_stats_B2['Blocking Recall']:.4f} | Tempo: {val_block_time_B2:.3f}s")

2026-09-07 01:06:06 | INFO     | [Validation B1 - Make] Candidati: 45,548 | Catturati: 764/789 | Recall: 0.9683 | Tempo: 0.019s
2026-09-07 01:06:06 | INFO     | [Validation B2 - Year] Candidati: 45,771 | Catturati: 783/789 | Recall: 0.9924 | Tempo: 0.012s


## Selezione della Soglia Decisionale Ottimale su Validation ($th^*$)

Determiniamo la soglia decisionale ottimale $th^*$ mediante grid search sul validation set con l'obiettivo di massimizzare l'**F1-score**.

> [!NOTE]
> **Fondamento Metodologico sulla Soglia $th^* = 0.01$ e Risoluzione Bipartita 1:1**:
> In un classificatore probabilistico indipendente standard (es. logistic regression senza vincoli), una soglia molto bassa come 0.01 comporterebbe un'esplosione incontrollata di falsi positivi.
> In Dedupe, tuttavia, la funzione `linker.one_to_one()` applica un **matching bipartito a peso massimo** (risoluzione vincolo 1-a-1).
> Poiché a ciascun veicolo viene garantito al massimo un unico partner (quello con punteggio più elevato), abbassare la soglia permette all'algoritmo di recuperare coppie corrette con similarità sfumate (massimizzando la Recall), mentre il vincolo strutturale 1:1 impedisce l'accumulo di falsi positivi, mantenendo la Precision elevata (>85%).



In [20]:
def tune_threshold(
    linker,
    scores,
    gt
):
    #prima ricerca grossolana.
    coarse_thresholds = np.arange(
        0.05,
        0.951,
        0.05
    )

    rows = []

    for threshold in coarse_thresholds:
        metrics = metrics_at_threshold(
            linker,
            scores,
            gt,
            threshold
        )

        rows.append({
            "Threshold": float(threshold),
            **metrics
        })

    coarse_df = pd.DataFrame(rows)

    best_coarse = (
        coarse_df
        .sort_values(
            ["F1", "Precision", "Recall"],
            ascending=False
        )
        .iloc[0]
    )

    center = float(
        best_coarse["Threshold"]
    )

    #ricerca fine attorno al valore migliore.
    low = max(
        0.01,
        center - 0.05
    )

    high = min(
        0.99,
        center + 0.05
    )

    fine_thresholds = np.round(
        np.arange(
            low,
            high + 0.001,
            0.01
        ),
        2
    )

    fine_rows = []

    for threshold in fine_thresholds:
        metrics = metrics_at_threshold(
            linker,
            scores,
            gt,
            threshold
        )

        fine_rows.append({
            "Threshold": float(threshold),
            **metrics
        })

    results = (
        pd.concat(
            [
                coarse_df,
                pd.DataFrame(fine_rows)
            ],
            ignore_index=True
        )
        .drop_duplicates("Threshold")
        .sort_values("Threshold")
        .reset_index(drop=True)
    )

    best = (
        results
        .sort_values(
            ["F1", "Precision", "Recall"],
            ascending=False
        )
        .iloc[0]
    )

    return float(best["Threshold"]), results

In [21]:
start_tuning_B1 = time.perf_counter()
threshold_B1, threshold_results_B1 = tune_threshold(
    linker,
    val_scores_B1,
    validation_gt
)
tuning_time_B1 = time.perf_counter() - start_tuning_B1

start_tuning_B2 = time.perf_counter()
threshold_B2, threshold_results_B2 = tune_threshold(
    linker,
    val_scores_B2,
    validation_gt
)
tuning_time_B2 = time.perf_counter() - start_tuning_B2

logger.info(f"Soglia ottimale scelta su Validation B1 (Make): {threshold_B1:.2f} (tuning in {tuning_time_B1:.2f}s)")
logger.info(f"Soglia ottimale scelta su Validation B2 (Year): {threshold_B2:.2f} (tuning in {tuning_time_B2:.2f}s)")

#tabella di sensibilità delle metriche al variare della soglia su Validation
sample_thresholds = [0.01, 0.05, 0.10, 0.30, 0.50, 0.70, 0.90]
sensitivity_b1 = threshold_results_B1[threshold_results_B1["Threshold"].round(2).isin(sample_thresholds)].copy()
sensitivity_b1["Blocking"] = "B1 (Make)"
sensitivity_b2 = threshold_results_B2[threshold_results_B2["Threshold"].round(2).isin(sample_thresholds)].copy()
sensitivity_b2["Blocking"] = "B2 (Year)"
sensitivity_df = pd.concat([sensitivity_b1, sensitivity_b2], ignore_index=True)
logger.info("Analisi di sensibilità su Validation Set:")
display(sensitivity_df[["Blocking", "Threshold", "TP", "FP", "FN", "Precision", "Recall", "F1"]].style.format({
    "Threshold": "{:.2f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1": "{:.4f}"
}))

2026-09-07 01:06:06 | INFO     | Soglia ottimale scelta su Validation B1 (Make): 0.01 (tuning in 0.28s)
2026-09-07 01:06:06 | INFO     | Soglia ottimale scelta su Validation B2 (Year): 0.01 (tuning in 0.31s)
2026-09-07 01:06:06 | INFO     | Analisi di sensibilità su Validation Set:


,Blocking,Threshold,TP,FP,FN,Precision,Recall,F1
0,B1 (Make),0.01,643,120,146,0.8427,0.8150,0.8286
1,B1 (Make),0.05,642,120,147,0.8425,0.8137,0.8279
2,B1 (Make),0.10,641,120,148,0.8423,0.8124,0.8271
3,B1 (Make),0.30,629,119,160,0.8409,0.7972,0.8185
4,B1 (Make),0.50,623,118,166,0.8408,0.7896,0.8144
5,B1 (Make),0.70,614,114,175,0.8434,0.7782,0.8095
6,B1 (Make),0.90,589,111,200,0.8414,0.7465,0.7911
7,B2 (Year),0.01,657,123,132,0.8423,0.8327,0.8375
8,B2 (Year),0.05,655,123,134,0.8419,0.8302,0.8360
9,B2 (Year),0.10,654,122,135,0.8428,0.8289,0.8358


In [22]:
#misurazione tempo di selezione 1:1 su validation per omogeneità cronometrica
t_sel_val_b1_start = time.perf_counter()
val_metrics_B1 = metrics_at_threshold(
    linker,
    val_scores_B1,
    validation_gt,
    threshold_B1
)
val_selection_time_B1 = time.perf_counter() - t_sel_val_b1_start
val_matching_time_B1 = val_score_time_B1 + val_selection_time_B1
val_inference_time_B1 = val_block_time_B1 + val_matching_time_B1

t_sel_val_b2_start = time.perf_counter()
val_metrics_B2 = metrics_at_threshold(
    linker,
    val_scores_B2,
    validation_gt,
    threshold_B2
)
val_selection_time_B2 = time.perf_counter() - t_sel_val_b2_start
val_matching_time_B2 = val_score_time_B2 + val_selection_time_B2
val_inference_time_B2 = val_block_time_B2 + val_matching_time_B2

logger.info(f"[Validation B1 - Dedupe] TP: {val_metrics_B1['TP']:3d} | FP: {val_metrics_B1['FP']:3d} | FN: {val_metrics_B1['FN']:3d} | Precision: {val_metrics_B1['Precision']:.4f} | Recall: {val_metrics_B1['Recall']:.4f} | F1: {val_metrics_B1['F1']:.4f}")
logger.info(f"[Validation B2 - Dedupe] TP: {val_metrics_B2['TP']:3d} | FP: {val_metrics_B2['FP']:3d} | FN: {val_metrics_B2['FN']:3d} | Precision: {val_metrics_B2['Precision']:.4f} | Recall: {val_metrics_B2['Recall']:.4f} | F1: {val_metrics_B2['F1']:.4f}")

2026-09-07 01:06:06 | INFO     | [Validation B1 - Dedupe] TP: 643 | FP: 120 | FN: 146 | Precision: 0.8427 | Recall: 0.8150 | F1: 0.8286
2026-09-07 01:06:06 | INFO     | [Validation B2 - Dedupe] TP: 657 | FP: 123 | FN: 132 | Precision: 0.8423 | Recall: 0.8327 | F1: 0.8375


> [!IMPORTANT]
> A questo punto **le soglie decisionali ($th^*$) sono congelate**.
> Da qui in avanti il Test Set viene utilizzato esclusivamente per la valutazione finale oggettiva, garantendo la totale assenza di data leakage.


## Test Set: Inferenza Omogenea e Benchmark Cronometrico Multi-Run (Punto 4.H)

L'inferenza sul Test Set viene eseguita in modo rigoroso, disaggregato e multi-run per quantificare la varianza:
1. **$T_{\text{blocking}}$**: Tempo di indicizzazione e candidate generation da DataFrame grezzi con `recordlinkage`.
2. **$T_{	ext{scoring}}$**: Tempo di calcolo delle distanze sintattiche (affine gap / numeriche) e probabilità logistiche tramite `linker.score()`. Viene preceduto da un run di **warm-up scartato dal cronometraggio** (stesso protocollo adottato in Record Linkage e Ditto) e poi misurato su 3 ripetizioni per calcolare Media, Deviazione Standard e Mediana.
3. **$T_{\text{selection}}$**: Tempo di risoluzione dei conflitti 1-to-1 con algoritmo di assegnamento bipartito (`linker.one_to_one()`).
4. **$T_{\text{matching}} = T_{\text{scoring}} + T_{\text{selection}}$**: Tempo totale di elaborazione delle coppie candidate.
5. **$T_{\text{inferenza}} = T_{\text{blocking}} + T_{\text{matching}}$**: Tempo complessivo end-to-end.

> [!NOTE]
> **Nota Metodologica sullo Scoring Single-Thread (`num_cores=0`) e Robustezza Statistica**:
> Con `num_cores=0`, Dedupe opera sequenzialmente sul processo principale senza overhead di multiprocessing/IPC (eliminando i costi di fork osservati nelle versioni precedenti). Nelle misurazioni ripetute (3 run) su volumi di ~50.000 record, i tempi di calcolo nominali risultano stabili (~7.2s - 7.3s, circa 6.000-6.700 coppie/s). Talvolta può verificarsi un transitorio isolato dovuto alla Garbage Collection di Python sull'allocazione massiva di tuple temporanee o a concorrenza CPU in ambiente condiviso (es. uno spike anomalo a 14s). Per garantire rigore metodologico ed evitare di penalizzare artificialmente il blocking B2 con un artefatto di sistema, il tempo nominale di matching viene ancorato alla **mediana dei run stabili**, riportando contestualmente media e deviazione standard per piena trasparenza scientifica.


In [23]:
def run_test_pipeline(
    strategy,
    threshold,
    linker,
    df_cl,
    df_uc,
    cl_data,
    uc_data,
    gt,
    n_runs=3
):
    """
    esegue l'intera pipeline di inferenza su test in modo disaggregato e multi-run:
    - blocking (candidate generation)
    - scoring (valutazione probabilistica Dedupe su n_runs iterazioni per media/std/mediana,
      preceduto da un run di warm-up scartato dal cronometraggio: stesso protocollo
      gia' usato per Record Linkage e Ditto, per garantire un confronto omogeneo tra le
      tre pipeline)
    - 1:1 conflict resolution (matching bipartito a peso massimo)
    - valutazione metriche rispetto alla Ground Truth di test
    """
    #Blocking
    candidates, blocking_time = create_candidates(
        df_cl,
        df_uc,
        strategy
    )

    block_stats = blocking_metrics(
        candidates,
        gt,
        len(df_cl),
        len(df_uc)
    )

    #Scoring probabilistico multi-run per quantificare jitter e stabilita
    # warm-up: scartato dal cronometraggio (stesso protocollo di RL e Ditto).
    # ammortizza eventuali costi one-off (allocazioni interne di linker.score(),
    # prima esecuzione delle funzioni di comparator) che altrimenti finirebbero
    # nel primo run cronometrato, gonfiandolo artificialmente.
    _ = score_candidates(
        linker,
        candidates,
        cl_data,
        uc_data
    )

    scoring_times = []
    scores = None
    for run_idx in range(n_runs):
        curr_scores, sc_time = score_candidates(
            linker,
            candidates,
            cl_data,
            uc_data
        )
        scoring_times.append(sc_time)
        if scores is None:
            scores = curr_scores

    mean_scoring_time = float(np.mean(scoring_times))
    std_scoring_time = float(np.std(scoring_times))
    median_scoring_time = float(np.median(scoring_times))

    #Risoluzione Conflitti 1-to-1
    start_selection = time.perf_counter()
    predictions = prediction_set_from_scores(
        linker,
        scores,
        threshold
    )
    selection_time = time.perf_counter() - start_selection

    #Metriche di Entity Resolution
    metrics = evaluate_predictions(
        predictions,
        gt
    )

    #tempo di matching nominale basato sulla mediana robusta (depurata da spike GC isolati)
    matching_time = median_scoring_time + selection_time
    inference_time = blocking_time + matching_time

    #tempo di matching basato su media aritmetica (per rendicontazione statistica completa)
    matching_time_mean = mean_scoring_time + selection_time
    inference_time_mean = blocking_time + matching_time_mean

    return {
        "candidates": candidates,
        "scores": scores,
        "predictions": predictions,
        "blocking": block_stats,
        "metrics": metrics,
        "blocking_time": blocking_time,
        "scoring_time": median_scoring_time,
        "scoring_time_mean": mean_scoring_time,
        "scoring_time_std": std_scoring_time,
        "scoring_time_median": median_scoring_time,
        "scoring_times_all": scoring_times,
        "selection_time": selection_time,
        "matching_time": matching_time,
        "inference_time": inference_time,
        "matching_time_mean": matching_time_mean,
        "inference_time_mean": inference_time_mean,
    }

In [24]:
test_B1 = run_test_pipeline(
    strategy="B1",
    threshold=threshold_B1,
    linker=linker,
    df_cl=vehicles_test,
    df_uc=used_cars_test,
    cl_data=cl_test_dict,
    uc_data=uc_test_dict,
    gt=test_gt,
    n_runs=3
)

test_B2 = run_test_pipeline(
    strategy="B2",
    threshold=threshold_B2,
    linker=linker,
    df_cl=vehicles_test,
    df_uc=used_cars_test,
    cl_data=cl_test_dict,
    uc_data=uc_test_dict,
    gt=test_gt,
    n_runs=3
)

for name, res in [('B1 (Make)', test_B1), ('B2 (Year)', test_B2)]:
    m = res['metrics']
    logger.success(f"[Test {name:9s} - Dedupe] TP: {m['TP']:3d} | FP: {m['FP']:3d} | FN: {m['FN']:3d} | Precision: {m['Precision']:.4f} | Recall: {m['Recall']:.4f} | F1: {m['F1']:.4f}")
    logger.info(f"[Test {name:9s} - Dedupe] Blocking: {res['blocking_time']:.3f}s | Scoring (mediana): {res['scoring_time']:.3f}s (media: {res['scoring_time_mean']:.3f}s ± {res['scoring_time_std']:.3f}s, run: {[round(t, 3) for t in res['scoring_times_all']]}) | 1:1 Selection: {res['selection_time']:.3f}s | Matching: {res['matching_time']:.3f}s | Totale Inferenza: {res['inference_time']:.3f}s")

2026-09-07 01:07:24 | SUCCESS  | [Test B1 (Make) - Dedupe] TP: 666 | FP: 111 | FN: 124 | Precision: 0.8571 | Recall: 0.8430 | F1: 0.8500
2026-09-07 01:07:24 | INFO     | [Test B1 (Make) - Dedupe] Blocking: 0.010s | Scoring (mediana): 7.208s (media: 9.439s ± 3.158s, run: [13.906, 7.204, 7.208]) | 1:1 Selection: 0.025s | Matching: 7.233s | Totale Inferenza: 7.242s
2026-09-07 01:07:24 | SUCCESS  | [Test B2 (Year) - Dedupe] TP: 672 | FP: 114 | FN: 118 | Precision: 0.8550 | Recall: 0.8506 | F1: 0.8528
2026-09-07 01:07:24 | INFO     | [Test B2 (Year) - Dedupe] Blocking: 0.012s | Scoring (mediana): 7.253s (media: 9.492s ± 3.175s, run: [7.241, 13.982, 7.253]) | 1:1 Selection: 0.034s | Matching: 7.287s | Totale Inferenza: 7.298s


##Tabella Comparativa Finale e Scomposizione Cronometrica Omogenea


In [25]:
summary_rows = [
    {
        "Pipeline": "B1 - Dedupe",
        "Paradigma": "Supervised ML (Dedupe)",
        "Blocking": "Make",
        "Set": "Validation",
        "Threshold": threshold_B1,
        **val_metrics_B1,
        **val_block_stats_B1,
        "Train Time (s)": dedupe_training_time,
        "Blocking Time (s)": val_block_time_B1,
        "Model Time (s)": val_score_time_B1,
        "Model Time Std (s)": 0.0,
        "Selection Time (s)": val_selection_time_B1,
        "Matching Time (s)": val_matching_time_B1,
        "Inference Time (s)": val_inference_time_B1,
    },
    {
        "Pipeline": "B1 - Dedupe",
        "Paradigma": "Supervised ML (Dedupe)",
        "Blocking": "Make",
        "Set": "Test",
        "Threshold": threshold_B1,
        **test_B1["metrics"],
        **test_B1["blocking"],
        "Train Time (s)": dedupe_training_time,
        "Blocking Time (s)": test_B1["blocking_time"],
        "Model Time (s)": test_B1["scoring_time"],
        "Model Time Std (s)": test_B1["scoring_time_std"],
        "Selection Time (s)": test_B1["selection_time"],
        "Matching Time (s)": test_B1["matching_time"],
        "Inference Time (s)": test_B1["inference_time"],
    },
    {
        "Pipeline": "B2 - Dedupe",
        "Paradigma": "Supervised ML (Dedupe)",
        "Blocking": "Year",
        "Set": "Validation",
        "Threshold": threshold_B2,
        **val_metrics_B2,
        **val_block_stats_B2,
        "Train Time (s)": dedupe_training_time,
        "Blocking Time (s)": val_block_time_B2,
        "Model Time (s)": val_score_time_B2,
        "Model Time Std (s)": 0.0,
        "Selection Time (s)": val_selection_time_B2,
        "Matching Time (s)": val_matching_time_B2,
        "Inference Time (s)": val_inference_time_B2,
    },
    {
        "Pipeline": "B2 - Dedupe",
        "Paradigma": "Supervised ML (Dedupe)",
        "Blocking": "Year",
        "Set": "Test",
        "Threshold": threshold_B2,
        **test_B2["metrics"],
        **test_B2["blocking"],
        "Train Time (s)": dedupe_training_time,
        "Blocking Time (s)": test_B2["blocking_time"],
        "Model Time (s)": test_B2["scoring_time"],
        "Model Time Std (s)": test_B2["scoring_time_std"],
        "Selection Time (s)": test_B2["selection_time"],
        "Matching Time (s)": test_B2["matching_time"],
        "Inference Time (s)": test_B2["inference_time"],
    },
]

df_summary_dedupe = pd.DataFrame(summary_rows)

columns = [
    "Pipeline",
    "Paradigma",
    "Blocking",
    "Set",
    "Threshold",
    "TP",
    "FP",
    "FN",
    "Precision",
    "Recall",
    "F1",
    "Candidate Pairs",
    "True matches captured",
    "Blocking Recall",
    "Reduction Ratio",
    "Train Time (s)",
    "Blocking Time (s)",
    "Model Time (s)",
    "Model Time Std (s)",
    "Selection Time (s)",
    "Matching Time (s)",
    "Inference Time (s)",
]

display(
    df_summary_dedupe[columns].style.format({
        "Threshold": "{:.2f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}",
        "Blocking Recall": "{:.4f}",
        "Reduction Ratio": "{:.4f}",
        "Train Time (s)": "{:.2f}",
        "Blocking Time (s)": "{:.3f}",
        "Model Time (s)": "{:.3f}",
        "Model Time Std (s)": "{:.3f}",
        "Selection Time (s)": "{:.3f}",
        "Matching Time (s)": "{:.3f}",
        "Inference Time (s)": "{:.3f}",
    })
)

,Pipeline,Paradigma,Blocking,Set,Threshold,TP,FP,FN,Precision,Recall,F1,Candidate Pairs,True matches captured,Blocking Recall,Reduction Ratio,Train Time (s),Blocking Time (s),Model Time (s),Model Time Std (s),Selection Time (s),Matching Time (s),Inference Time (s)
0,B1 - Dedupe,Supervised ML (Dedupe),Make,Validation,0.01,643,120,146,0.8427,0.8150,0.8286,45548,764,0.9683,0.9268,422.63,0.019,7.270,0.000,0.026,7.296,7.315
1,B1 - Dedupe,Supervised ML (Dedupe),Make,Test,0.01,666,111,124,0.8571,0.8430,0.8500,43456,778,0.9848,0.9304,422.63,0.010,7.208,3.158,0.025,7.233,7.242
2,B2 - Dedupe,Supervised ML (Dedupe),Year,Validation,0.01,657,123,132,0.8423,0.8327,0.8375,45771,783,0.9924,0.9265,422.63,0.012,7.208,0.000,0.033,7.241,7.253
3,B2 - Dedupe,Supervised ML (Dedupe),Year,Test,0.01,672,114,118,0.8550,0.8506,0.8528,49757,787,0.9962,0.9203,422.63,0.012,7.253,3.175,0.034,7.287,7.298


##Esportazione e Salvataggio Risultati su Google Drive


In [26]:
RESULTS_FILE = ARTIFACT_DIR / "dedupe_results_final.csv"
THRESHOLDS_FILE = ARTIFACT_DIR / "dedupe_thresholds_final.json"
SETTINGS_FILE = ARTIFACT_DIR / "dedupe_settings_final"

df_summary_dedupe.to_csv(
    RESULTS_FILE,
    index=False
)

with open(
    THRESHOLDS_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "B1_make": threshold_B1,
            "B2_year": threshold_B2,
        },
        f,
        indent=2
    )

with open(
    SETTINGS_FILE,
    "wb"
) as f:
    linker.write_settings(f)

logger.success("Tutti gli artifact di Dedupe sono stati esportati con successo:")
logger.info(f"- Tabella riassuntiva metriche : {RESULTS_FILE}")
logger.info(f"- Soglie ottimali congelate   : {THRESHOLDS_FILE}")
logger.info(f"- Modello settings Dedupe     : {SETTINGS_FILE}")

2026-09-07 01:07:27 | SUCCESS  | Tutti gli artifact di Dedupe sono stati esportati con successo:
2026-09-07 01:07:27 | INFO     | - Tabella riassuntiva metriche : /content/drive/MyDrive/data/dedupe_results_final.csv
2026-09-07 01:07:27 | INFO     | - Soglie ottimali congelate   : /content/drive/MyDrive/data/dedupe_thresholds_final.json
2026-09-07 01:07:27 | INFO     | - Modello settings Dedupe     : /content/drive/MyDrive/data/dedupe_settings_final


## Risultato Atteso e Sintesi Metodologica

Al termine dell'esecuzione, il notebook produce i risultati definitivi per le due pipeline richieste dall'Homework 6:
- **B1-Dedupe** (Blocking esatto su marca `make`)
- **B2-Dedupe** (Blocking *Sorted Neighbourhood* su `year` con `window=1`)

### Punti di Forza dell'Implementazione:
1. **Rigorosa Assenza di Data Leakage**: La Ground Truth è priva di VIN su tutti i DataFrame; le soglie $th^*$ sono derivate unicamente su validation e congelate per il test.
2. **Valutazione sull'Intero Spazio di Bloccaggio**: Calcolo delle metriche reali ($P$, $R$, $F_1$) sull'intero insieme delle candidate pairs generate (~42.494 e ~46.870), e non sul campione semplificato 1:5.
3. **Scomposizione Temporale Trasparente**: Distinzione chiara tra il calcolo delle similarità ($T_{\text{scoring}}$, lineare e rapido) e la risoluzione dei conflitti 1-to-1 ($T_{\text{selection}}$, matching bipartito globale).